# Market-Adjusted Target And Surprise-Feature Comparison

This notebook compares two target definitions and two alt-data representations on the same `TSLA + AAPL` panel.

Targets:
- `raw_direction`
- `excess_direction`

Feature representations:
- `price_only`
- `price_plus_alt_current`
- `price_plus_alt_surprise`

Models:
- `logreg`
- `hist_gradient_boosting`

Benchmark handling:
- first try local market benchmark files like `SPY/QQQ`
- otherwise fall back to an equal-weight proxy built from the panel itself

Artifacts are written to `notebooks_two_tickers/outputs/market_adjusted_surprise`.


In [1]:
import pandas as pd

from market_adjusted_surprise_experiment import run_market_adjusted_surprise_experiment

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 260)


In [2]:
result = run_market_adjusted_surprise_experiment()

print(f"Dataset: {result['dataset_path']}")
print(f"Metrics: {result['metrics_path']}")
print(f"Predictions: {result['predictions_path']}")
print(f"Metadata: {result['metadata_path']}")
print(f"Feature sets: {result['feature_sets_path']}")
print(f"Benchmark source: {result['benchmark_metadata']['benchmark_source']}")
if result['benchmark_metadata']['benchmark_path']:
    print(f"Benchmark path: {result['benchmark_metadata']['benchmark_path']}")


Dataset: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\datasets\stock_direction_two_tickers_base.csv
Metrics: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\market_adjusted_surprise\metrics.csv
Predictions: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\market_adjusted_surprise\predictions.csv
Metadata: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\market_adjusted_surprise\run_metadata.json
Feature sets: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\market_adjusted_surprise\feature_sets.json
Benchmark source: equal_weight_panel_proxy


In [3]:
test_all = result["metrics"].query("split == 'test' and ticker == 'ALL'").copy()
test_all = test_all.sort_values(["target_mode", "model_name", "feature_representation"]).reset_index(drop=True)
test_all


,ticker,model_name,feature_representation,target_mode,split,n_rows,n_features,accuracy,balanced_accuracy,f1,roc_auc
0,ALL,hist_gradient_boosting,price_only,excess_direction,test,256,14,0.531250,0.531250,0.531250,0.511353
1,ALL,hist_gradient_boosting,price_plus_alt_current,excess_direction,test,256,32,0.511719,0.511719,0.517375,0.515259
2,ALL,hist_gradient_boosting,price_plus_alt_surprise,excess_direction,test,256,33,0.480469,0.480469,0.470120,0.498901
3,ALL,logreg,price_only,excess_direction,test,256,14,0.550781,0.550781,0.534413,0.577087
4,ALL,logreg,price_plus_alt_current,excess_direction,test,256,32,0.562500,0.562500,0.525424,0.575439
5,ALL,logreg,price_plus_alt_surprise,excess_direction,test,256,33,0.535156,0.535156,0.533333,0.570496
6,ALL,hist_gradient_boosting,price_only,raw_direction,test,256,14,0.562500,0.558569,0.600000,0.556232
7,ALL,hist_gradient_boosting,price_plus_alt_current,raw_direction,test,256,32,0.519531,0.520353,0.535849,0.520261
8,ALL,hist_gradient_boosting,price_plus_alt_surprise,raw_direction,test,256,33,0.531250,0.528439,0.565217,0.526164
9,ALL,logreg,price_only,raw_direction,test,256,14,0.554688,0.560844,0.544000,0.559860


In [4]:
summary = test_all.pivot_table(
    index=["target_mode", "model_name"],
    columns="feature_representation",
    values=["balanced_accuracy", "roc_auc"],
)
summary


balanced_accuracy                                                   roc_auc                                               
feature_representation                         price_only price_plus_alt_current price_plus_alt_surprise price_only price_plus_alt_current price_plus_alt_surprise
target_mode      model_name                                                                                                                                       
excess_direction hist_gradient_boosting          0.531250               0.511719                0.480469   0.511353               0.515259                0.498901
                 logreg                          0.550781               0.562500                0.535156   0.577087               0.575439                0.570496
raw_direction    hist_gradient_boosting          0.558569               0.520353                0.528439   0.556232               0.520261                0.526164
                 logreg                          0.560844               0.496526                0.526656   0.559860               0.510115                0.526902

In [5]:
test_by_ticker = result["metrics"].query("split == 'test' and ticker != 'ALL'").copy()
test_by_ticker = test_by_ticker.sort_values(["ticker", "target_mode", "model_name", "feature_representation"]).reset_index(drop=True)
test_by_ticker


,ticker,model_name,feature_representation,target_mode,split,n_rows,n_features,accuracy,balanced_accuracy,f1,roc_auc
0,AAPL,hist_gradient_boosting,price_only,excess_direction,test,128,14,0.539062,0.542766,0.581560,0.515152
1,AAPL,hist_gradient_boosting,price_plus_alt_current,excess_direction,test,128,32,0.546875,0.547898,0.553846,0.546188
2,AAPL,hist_gradient_boosting,price_plus_alt_surprise,excess_direction,test,128,33,0.437500,0.439394,0.462687,0.442815
3,AAPL,logreg,price_only,excess_direction,test,128,14,0.554688,0.555474,0.558140,0.588954
4,AAPL,logreg,price_plus_alt_current,excess_direction,test,128,32,0.625000,0.622190,0.578947,0.603128
5,AAPL,logreg,price_plus_alt_surprise,excess_direction,test,128,33,0.570312,0.569648,0.552846,0.596285
6,AAPL,hist_gradient_boosting,price_only,raw_direction,test,128,14,0.546875,0.533990,0.618421,0.552709
7,AAPL,hist_gradient_boosting,price_plus_alt_current,raw_direction,test,128,32,0.507812,0.505665,0.540146,0.508867
8,AAPL,hist_gradient_boosting,price_plus_alt_surprise,raw_direction,test,128,33,0.500000,0.492611,0.555556,0.487931
9,AAPL,logreg,price_only,raw_direction,test,128,14,0.507812,0.511576,0.511628,0.497044
